# BERT (Bidirectional Encoder Representations from Transformers)

**Domain:** Architectures  ·  **from study list**  ·  **runnable:** yes

## 1. What & Why

**BERT** is a Transformer **encoder** pre-trained to read text *bidirectionally* — every
token attends to the whole sentence at once, left and right. It was the 2018 model that made
"pre-train once, fine-tune everywhere" the default recipe for NLP.

The problem it solves: before BERT, language models were trained left-to-right (GPT) or as two
separate left-to-right and right-to-left passes (ELMo). That hobbles tasks where meaning depends
on *both* sides of a word ("bank" in *river bank* vs *bank account*). BERT trains on a
**fill-in-the-blank** objective so a token's representation fuses context from both directions
in every layer.

**Reach for BERT when** you need a strong text *encoder* — classification, named-entity
recognition, extractive QA (span selection), sentence-pair scoring, or dense retrieval
embeddings. You fine-tune the pre-trained weights on a few thousand labeled examples and get
near-SOTA results cheaply.

**Don't reach for it when** you need to *generate* fluent text. BERT has no decoder and was never
trained to produce sequences token-by-token; use a decoder (GPT-style) or encoder-decoder (T5,
BART) for generation. See [`encoder-decoder`](encoder-decoder.ipynb) and [`t5`](t5.ipynb).

## 2. Mental Model

**A student doing cloze (fill-in-the-blank) exercises.** You hand them a sentence with ~15% of
the words blanked out and they must guess each blank using *everything else* in the sentence:

```
input :  the  [MASK]  sat  on  the  [MASK]
guess :         cat                  mat
```

Because the blank can be anywhere, the student must build a representation of every word that
already "knows" about its neighbors on both sides. That is exactly what a stack of bidirectional
self-attention layers computes.

Contrast with a GPT-style decoder, which reads with a blindfold that only lets it see words to
the *left* (causal masking) so it can predict the *next* word. BERT removes the blindfold — which
is why it can't generate, but reads better.

## 3. Key Concepts

- **Encoder-only Transformer.** A stack of identical blocks (base = 12 layers, 110M params;
  large = 24 layers, 340M). Each block = bidirectional multi-head self-attention → add & LayerNorm
  → feed-forward → add & LayerNorm. No causal mask, no decoder, no cross-attention.

- **Masked Language Modeling (MLM).** The pre-training objective. Randomly pick 15% of tokens;
  of those, replace **80% with `[MASK]`, 10% with a random token, 10% left unchanged**, then
  predict the originals. The 10/10 trick stops the model from only ever "seeing" `[MASK]` and
  forces it to keep a useful representation of *every* token (since `[MASK]` never appears at
  fine-tune/inference time).

- **Next Sentence Prediction (NSP).** A secondary objective: given two segments, predict whether
  B actually followed A. Later work (RoBERTa) showed NSP is largely unnecessary and dropped it.

- **Special tokens & segments.** Every input starts with **`[CLS]`** (its final hidden state is
  the pooled sentence representation used for classification) and uses **`[SEP]`** to separate
  two sentences. A learned **segment embedding** (A/B) is added so the model knows which sentence
  a token belongs to.

- **WordPiece tokenization.** Subword vocabulary (~30k). Unknown/rare words split into pieces
  with `##` continuation markers: `playing → play ##ing`. Keeps the vocabulary small while
  avoiding out-of-vocabulary tokens.

- **Three summed embeddings.** Token + positional (learned, absolute, max length 512) + segment.
  The sum is the input to layer 1.

- **Fine-tuning.** Add a tiny task head (usually a single linear layer) on top of `[CLS]` or the
  per-token outputs, then train end-to-end with a small learning rate (~2e-5). Cheap: minutes to
  hours on one GPU.

## 4. Setup

The first two examples are **pure PyTorch** and run on CPU in under a second — no downloads.
Example 3 loads the real `bert-base-uncased` weights from Hugging Face (~440 MB) and is gated
behind an environment variable so the notebook executes cleanly offline.

In [1]:
# %pip install torch transformers
import torch
import torch.nn as nn
import transformers

torch.manual_seed(0)
print("torch", torch.__version__, "| transformers", transformers.__version__)

/Users/danieldekerlegand/Development/ai-tutor/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch 2.12.1 | transformers 5.12.1


## 5. Worked Examples

### Example 1 — One bidirectional encoder block, and *proving* it's bidirectional

A BERT layer is just self-attention with **no causal mask**, so every position can see every
other position. We build one tiny block, run a forward pass, then perturb the *last* token and
watch the *first* token's output change — something a left-to-right model could never do.

In [2]:
import torch.nn.functional as F

VOCAB, D_MODEL, N_HEADS, SEQ = 30, 16, 2, 6

class EncoderBlock(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        self.attn = nn.MultiheadAttention(d, h, batch_first=True)
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.ff = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))

    def forward(self, x):
        a, _ = self.attn(x, x, x, need_weights=False)  # no attn_mask => bidirectional
        x = self.ln1(x + a)                            # residual + LayerNorm
        x = self.ln2(x + self.ff(x))
        return x

tok = torch.randint(0, VOCAB, (1, SEQ))
emb = nn.Embedding(VOCAB, D_MODEL)
pos = nn.Embedding(SEQ, D_MODEL)
x = emb(tok) + pos(torch.arange(SEQ))          # token + positional embeddings

block = EncoderBlock(D_MODEL, N_HEADS).eval()
out = block(x)
print("token ids        :", tok[0].tolist())
print("input embeddings :", tuple(x.shape), "-> contextual output:", tuple(out.shape))

# Bidirectionality: perturb the LAST token, measure how token 0's output moves.
x2 = x.clone()
x2[0, -1] += 1.0
delta = (block(x)[0, 0] - block(x2)[0, 0]).abs().max().item()
print(f"token 0's representation shifts by {delta:.4f} when the LAST token changes "
      f"-> it reads rightward")

token ids        : [14, 9, 23, 0, 13, 9]
input embeddings : (1, 6, 16) -> contextual output: (1, 6, 16)
token 0's representation shifts by 0.0196 when the LAST token changes -> it reads rightward


### Example 2 — The MLM masking recipe (the 80/10/10 rule)

This is the data corruption that makes BERT work. Pick 15% of positions to predict; of those,
80% become `[MASK]`, 10% become a random token, 10% stay unchanged. Non-selected positions are
set to `-100` in the labels so PyTorch's cross-entropy ignores them.

In [3]:
torch.manual_seed(1)

MASK_ID = 103                       # bert-base-uncased's [MASK] id
ids = torch.arange(10, 20)          # a toy 10-token "sentence"
labels = ids.clone()

# 1) select 15% of positions to predict
selected = torch.bernoulli(torch.full(ids.shape, 0.15)).bool()
labels[~selected] = -100            # ignore everything we didn't select

# 2) of the selected: 80% -> [MASK], 10% -> random token, 10% -> unchanged
inp = ids.clone()
r = torch.rand(ids.shape)
inp[selected & (r < 0.8)] = MASK_ID
rand = selected & (r >= 0.8) & (r < 0.9)
inp[rand] = torch.randint(0, 30000, ids.shape)[rand]

print("original :", ids.tolist())
print("model in :", inp.tolist(), "  (103 = [MASK])")
print("labels   :", labels.tolist(), "  (-100 = ignored by the loss)")
print("predicted positions:", selected.nonzero().flatten().tolist())

original : [10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
model in : [10, 11, 12, 13, 103, 15, 16, 17, 18, 19]   (103 = [MASK])
labels   : [-100, -100, -100, -100, 14, -100, -100, -100, -100, -100]   (-100 = ignored by the loss)
predicted positions: [4]


### Example 3 — Real `bert-base-uncased` filling a blank (gated)

The pre-trained model genuinely does the cloze task. This cell downloads ~440 MB the first time,
so it only runs when `RUN_HF=1`. The call shape is shown either way.

In [4]:
import os

if os.getenv("RUN_HF"):
    from transformers import pipeline
    fill = pipeline("fill-mask", model="bert-base-uncased")
    for r in fill("The capital of France is [MASK].")[:3]:
        print(f"{r['score']:.3f}  {r['token_str']!r}")
else:
    print("Set RUN_HF=1 to download bert-base-uncased (~440MB) and run fill-mask.")
    print('Call shape:')
    print('  from transformers import pipeline')
    print('  fill = pipeline("fill-mask", model="bert-base-uncased")')
    print('  fill("The capital of France is [MASK].")')
    print('Expected top guess: "paris"')

Set RUN_HF=1 to download bert-base-uncased (~440MB) and run fill-mask.
Call shape:
  from transformers import pipeline
  fill = pipeline("fill-mask", model="bert-base-uncased")
  fill("The capital of France is [MASK].")
Expected top guess: "paris"


## 6. Gotchas & Pitfalls

- **It cannot generate text.** No decoder, no autoregressive training. Filling a single `[MASK]`
  is not generation. For generation use GPT (decoder) or T5/BART (encoder-decoder).

- **512-token hard limit.** Positional embeddings are learned for positions 0–511. Longer inputs
  must be truncated or chunked, or use a long-context variant (Longformer, BigBird).

- **`[MASK]` never appears at inference.** That's the whole reason for the 80/10/10 split. If you
  fine-tune by feeding `[MASK]` tokens you create a train/serve mismatch the model wasn't built
  for — mask in the *pre-training* data, not in your downstream inputs.

- **The `[CLS]` vector is not a good sentence embedding off the shelf.** Raw BERT `[CLS]` (or mean
  pooling) gives weak semantic similarity. For sentence embeddings use **Sentence-BERT**, which
  fine-tunes with a siamese contrastive objective.

- **`uncased` lowercases and strips accents.** Fine for general English; use a `cased` checkpoint
  for NER, code, or anything where capitalization carries meaning.

- **Always use the matching tokenizer.** WordPiece vocab, special-token ids, and lowercasing are
  tied to the checkpoint. Load `AutoTokenizer.from_pretrained(same_name)` — never hand-roll ids.

- **Fine-tune gently.** Use a small LR (~2e-5–5e-5), 2–4 epochs, and warmup. Large LRs wreck the
  pre-trained weights ("catastrophic forgetting") and the run diverges or underperforms.

## 7. When to Use vs Alternatives

| Option | Pick it when… | Trade-off vs BERT |
|---|---|---|
| **BERT (encoder-only)** | Classification, NER, extractive QA, retrieval embeddings | Can't generate text |
| **RoBERTa / DeBERTa** | You want a drop-in stronger encoder | Same API; better pre-training (more data, no NSP, disentangled attention) — usually just use these |
| **DistilBERT / MobileBERT** | Latency/size matters | ~40% smaller, ~60% faster, a few points lower accuracy |
| **GPT (decoder-only)** | Text generation, chat, few-shot prompting | No bidirectional context; weaker as a pure encoder |
| **T5 / BART (encoder-decoder)** | Summarization, translation, seq2seq | Heavier; overkill if you only need an encoder |
| **Sentence-BERT** | Semantic similarity / clustering of sentences | Same backbone, fine-tuned for embeddings — vanilla BERT pooling is poor for this |

Rule of thumb: **understanding/labeling tasks → an encoder (start with DeBERTa/RoBERTa); producing
text → a decoder or encoder-decoder.** BERT remains the canonical encoder and a great default for
fine-tuned classifiers.

## 8. Resources

- **BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding** — Devlin et
  al., 2018, the original paper: https://arxiv.org/abs/1810.04805
- **The Illustrated BERT** — Jay Alammar's visual walkthrough: https://jalammar.github.io/illustrated-bert/
- **Hugging Face BERT docs** — model, tokenizer, and task heads: https://huggingface.co/docs/transformers/model_doc/bert
- **RoBERTa: A Robustly Optimized BERT Pretraining Approach** — Liu et al., 2019 (what to actually use): https://arxiv.org/abs/1907.11692
- **Sentence-BERT** — Reimers & Gurevych, 2019, for sentence embeddings: https://arxiv.org/abs/1908.10084